# Dal suono alle feature

Il codice del capitolo [«Dal suono alle feature»](https://book.paithon.it/main/Audio/dal-suono-alle-feature.html), *Paithon Book*.

Le celle sono quelle del libro, nell'ordine in cui compaiono: il testo che le spiega sta nelle pagine, qui c'è solo la parte da eseguire e da rompere.

Generato da `scripts/genera-notebook.py`: le correzioni vanno fatte nelle pagine del libro, non qui.


> **Verificato il 2026-07-25** con torch 2.13.0, numpy 2.4.6, pandas 3.0.5, scikit-learn 1.9.0, transformers 5.14.1, diffusers 0.39.0, librosa 0.11.0, torch-geometric 2.8.0.post1. Tutte le celle di questo notebook sono state eseguite senza errori con quelle versioni; le librerie si muovono, e se qualcosa qui non gira piu' e' un errore del libro: [segnalalo](https://github.com/paithon-it/paithonbook/issues).


In [ ]:
# Su Colab quasi tutto c'è già; questa riga serve altrove.
%pip install -q librosa numpy

In [ ]:
# Mostra il valore di ogni riga, come i commenti «# ->» del libro.
try:
    from IPython.core.interactiveshell import InteractiveShell
    InteractiveShell.ast_node_interactivity = 'all'
except ImportError:      # fuori da IPython non serve e non c'è
    pass

> **Cella di preparazione.** Crea i dati e i nomi che il testo da per esistenti. Non fa parte del libro: serve a far girare il notebook, e viene ripetuta all'inizio di ogni pagina perche ognuna riparta dallo stesso stato.


In [ ]:
_PRELUDIO = r'''
import numpy as np
import soundfile as sf

# Il file audio che la pagina dà per esistente. Mezzo secondo di parlato finto:
# una portante a 220 Hz modulata in ampiezza, che basta a far uscire uno
# spettrogramma con una struttura visibile invece di rumore piatto.
sr = 16_000
t = np.linspace(0, 0.5, int(0.5 * sr), endpoint=False)
inviluppo = 0.5 * (1 + np.sin(2 * np.pi * 4 * t))
onda = inviluppo * (np.sin(2 * np.pi * 220 * t) + 0.3 * np.sin(2 * np.pi * 660 * t))
sf.write("frase.wav", (0.4 * onda).astype(np.float32), sr)
'''
exec(_PRELUDIO)

## Dal suono alle feature

[Leggi la pagina](https://book.paithon.it/main/Audio/dal-suono-alle-feature.html)


### In pratica


In [ ]:
import librosa

# carica l'audio prendendo 16.000 misure al secondo (lo standard per la voce)
y, sr = librosa.load("frase.wav", sr=16000)

# spettrogramma mel: 40 bande, finestre da 25 ms ogni 10 ms
# htk=True sceglie una delle due scale mel in circolazione: danno bande
# diverse, quindi qual e' delle due va sempre dichiarato
S = librosa.feature.melspectrogram(
    y=y, sr=sr, n_fft=400, hop_length=160, n_mels=40, htk=True
)
# intensita' schiacciate (la parte "log" del log-mel): i suoni deboli
# tornano visibili accanto a quelli forti
S_db = librosa.power_to_db(S, ref=S.max())

# 13 coefficienti MFCC per ogni finestra temporale: il riassunto piu' corto.
# Si calcolano DALLA S_db appena ottenuta. Chiamando invece mfcc(y=...) librosa
# rifarebbe lo spettrogramma da capo con i propri default (n_fft=2048,
# hop_length=512), quindi su un asse dei tempi diverso dal nostro.
mfcc = librosa.feature.mfcc(S=S_db, n_mfcc=13)

print(S_db.shape, mfcc.shape)  # (bande, tempo) e (coefficienti, tempo)

## Riconoscere i suoni: classificazione e tagging

[Leggi la pagina](https://book.paithon.it/main/Audio/classificazione-audio.html)


In [ ]:
exec(_PRELUDIO)   # ripristina i nomi di partenza della pagina

### Un classificatore di suoni in miniatura


In [ ]:
import numpy as np

rng = np.random.default_rng(0)   # punto di partenza fissato: numeri sempre uguali
fs = 8000                        # frequenza di campionamento (Hz)
dur = 0.15                       # durata di ogni segmento (secondi)
n = int(fs * dur)                # campioni per segmento
t = np.arange(n) / fs

# Tre segmenti: un tono puro, del silenzio, del rumore
tono     = 1.0 * np.sin(2 * np.pi * 200 * t)        # sinusoide a 200 Hz
silenzio = 0.001 * rng.standard_normal(n)           # quasi-zero (fondo)
rumore   = 0.30 * rng.standard_normal(n)            # rumore gaussiano
segnale  = np.concatenate([tono, silenzio, rumore])

def energia(x):
    "Energia a breve termine: potenza media della finestra."
    return float(np.mean(x**2))

def zcr(x):
    "Zero-crossing rate: frazione di cambi di segno tra campioni adiacenti."
    return float(np.mean(np.abs(np.diff(np.sign(x)))) / 2)

L = 400  # finestra: 400 campioni che qui, a 8 kHz, fanno 50 ms
         # (nella prima sezione 400 campioni erano 25 ms perche' li' si
         #  misurava 16.000 volte al secondo invece di 8.000)
SOGLIA_E, SOGLIA_Z = 0.01, 0.20   # soglie di decisione

def classifica(e, z):
    if e < SOGLIA_E:              # poca energia: nessun suono
        return "silenzio"
    if z > SOGLIA_Z:              # tanti cambi di segno: rumore/sibilo
        return "rumore"
    return "tono"                 # energia alta, pochi cambi: tono/voce

print(f"{'finestra':>8} | {'energia':>9} | {'zcr':>6} | classe")
print("-" * 42)
for i in range(0, len(segnale) - L + 1, L):
    finestra = segnale[i:i+L]
    e, z = energia(finestra), zcr(finestra)
    print(f"{i//L:>8} | {e:>9.4f} | {z:>6.3f} | {classifica(e, z)}")

## Imparare dal suono senza etichette

[Leggi la pagina](https://book.paithon.it/main/Audio/rappresentazioni-auto-supervisionate.html)


In [ ]:
exec(_PRELUDIO)   # ripristina i nomi di partenza della pagina

### wav2vec 2.0: mascherare il suono


In [ ]:
import numpy as np

rng = np.random.default_rng(0)
d = 8  # quanti numeri ha ogni gruppetto (nei modelli veri sono centinaia)

# c: quello che il modello si e' fatto in mente del pezzetto COPERTO.
# In un modello ben addestrato e' vicino all'unita' giusta e lontano dai distrattori.
c = rng.standard_normal(d)

# q_true: l'unita' corretta (qui una versione "vicina" a c);
# i distrattori sono unita' pescate da altri pezzetti coperti della stessa frase.
q_true = c + 0.3 * rng.standard_normal(d)
q_dist = rng.standard_normal((4, d))
candidati = np.vstack([q_true, q_dist])      # (5, d): il vero piu' 4 distrattori

def coseno(a, B):                            # coseno tra a e ogni riga di B
    a = a / np.linalg.norm(a)
    B = B / np.linalg.norm(B, axis=1, keepdims=True)
    return B @ a

kappa = 0.1                                  # "temperatura": piu' e' bassa,
                                             # piu' la scelta esce netta
punteggi = coseno(c, candidati) / kappa
prob = np.exp(punteggi - punteggi.max())
prob /= prob.sum()                           # softmax: i punteggi riscalati
                                             # cosi' che sommino a uno

print("prob. per candidato:", prob.round(3))
print("scelto:", int(prob.argmax()), "(0 = unita' giusta)")

## Il suono come token: i codec neurali

[Leggi la pagina](https://book.paithon.it/main/Audio/codec-neurali.html)


In [ ]:
exec(_PRELUDIO)   # ripristina i nomi di partenza della pagina

### Un RVQ in miniatura


In [ ]:
import numpy as np

rng = np.random.default_rng(0)

# Sei pezzetti da due numeri ciascuno: quelli che uscirebbero dall'encoder
Z = rng.uniform(-1, 1, size=(6, 2)).round(2)

# Primo codebook: 4 prototipi grossolani (K = 4)
C1 = np.array([[-0.5, -0.5],
               [ 0.5, -0.5],
               [-0.5,  0.5],
               [ 0.5,  0.5]])

# Secondo codebook: 4 aggiustamenti fini per il residuo (lo zero e' incluso)
C2 = np.array([[ 0.00,  0.00],
               [ 0.30,  0.00],
               [ 0.00,  0.30],
               [-0.30, -0.30]])


def quantizza(V, C):
    """Per ogni riga di V trova il prototipo piu' vicino nell'elenco C."""
    # distanze quadratiche fra ogni pezzetto e ogni prototipo
    d = ((V[:, None, :] - C[None, :, :]) ** 2).sum(axis=2)
    idx = d.argmin(axis=1)      # posizione del prototipo piu' vicino: il "token"
    return idx, C[idx]          # le posizioni e i pezzetti arrotondati


def mse(A, B):
    """MSE, errore quadratico medio: di quanto sbaglia in media la ricostruzione."""
    return ((A - B) ** 2).mean()


# --- Stadio 1: arrotondo il pezzetto al prototipo piu' vicino ---
idx1, q1 = quantizza(Z, C1)
ric1 = q1                       # ricostruzione con 1 solo stadio

# --- Stadio 2: quantizzo il RESIDUO ---
residuo = Z - q1
idx2, q2 = quantizza(residuo, C2)
ric2 = q1 + q2                  # ricostruzione con 2 stadi

print("vettori da quantizzare:\n", Z)
print("token stadio 1:", idx1.tolist())
print("token stadio 2:", idx2.tolist())
print(f"MSE con 1 quantizzatore: {mse(Z, ric1):.4f}")
print(f"MSE con 2 quantizzatori: {mse(Z, ric2):.4f}")

## Generare suono e musica

[Leggi la pagina](https://book.paithon.it/main/Audio/generazione-audio.html)


In [ ]:
exec(_PRELUDIO)   # ripristina i nomi di partenza della pagina

### La via storica: WaveNet, campione per campione


In [ ]:
import numpy as np

mu = 255  # 8 bit -> 256 livelli, come nel WaveNet originale

def comprimi(x):                 # mu-law: schiaccia verso le ampiezze piccole
    return np.sign(x) * np.log1p(mu * np.abs(x)) / np.log1p(mu)

def espandi(y):                  # operazione inversa
    return np.sign(y) * ((1 + mu) ** np.abs(y) - 1) / mu

def a_8bit(v):                   # da [-1, 1] a 256 livelli interi e ritorno
    q = np.round((v + 1) / 2 * mu).astype(int)     # 0..255: un byte per campione
    return q / mu * 2 - 1

# segnale di prova: una nota che sfuma quasi al silenzio (ampia gamma dinamica)
t = np.linspace(0, 1, 16000, endpoint=False)       # 1 s a 16 kHz
x = (np.sin(2*np.pi*220*t) + 0.5*np.sin(2*np.pi*440*t)) * np.exp(-5*t)
x = x / np.max(np.abs(x))                          # normalizza in [-1, 1]

x_mulaw   = espandi(a_8bit(comprimi(x)))           # 8 bit CON compansione mu-law
x_lineare = a_8bit(x)                              # 8 bit SENZA (quantizzazione lineare)

def snr_segmentale(x, xh, win=320):                # SNR medio su finestre di 20 ms
    n = (len(x) // win) * win
    ps = np.sum(x[:n].reshape(-1, win)**2, axis=1)
    pe = np.sum((x[:n]-xh[:n]).reshape(-1, win)**2, axis=1)
    m = (ps > 1e-9) & (pe > 1e-12)                 # ignora i frame di puro silenzio
    return np.mean(10*np.log10(ps[m]/pe[m]))

print(f"errore massimo (mu-law):    {np.max(np.abs(x - x_mulaw)):.4f}")
print(f"errore massimo (lineare):   {np.max(np.abs(x - x_lineare)):.4f}")
print(f"SNR segmentale mu-law:      {snr_segmentale(x, x_mulaw):.1f} dB")
print(f"SNR segmentale lineare:     {snr_segmentale(x, x_lineare):.1f} dB")